In [11]:
from __future__ import annotations

import sys
import json
import time
import importlib.util
from pathlib import Path

# Resolve the checkout from either the project root or research/notebooks.
PROJECT_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "run.py").is_file() and (p / "backend").is_dir()
)

from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

import torch

# Dossier exact de ton projet
SCRIPT_DIR = PROJECT_ROOT

TABPFN_EMBEDDING_PATH = PROJECT_ROOT / "artifacts/experiments/tabpfn" / "tabpfn_embeddings.parquet"

PROPOS_CACHE_DIR = PROJECT_ROOT / ".local/cache/research/propos"
PROPOS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PROPOS_MASTER_DIR = PROJECT_ROOT / "vendor/propos"
PROPOS_TABULAR_ADAPTER_PATH = PROPOS_MASTER_DIR / "tabular_adapter.py"

RANDOM_STATE = 42
K = 5
EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

RUN_NAME = f"propos_tabpfn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

ARTIFACT_PATH = PROPOS_CACHE_DIR / f"{RUN_NAME}.pt"
ASSIGNMENTS_PATH = PROPOS_CACHE_DIR / f"{RUN_NAME}_assignments.parquet"
SUMMARY_PATH = PROPOS_CACHE_DIR / f"{RUN_NAME}_summary.json"

print("SCRIPT_DIR:", SCRIPT_DIR)
print("TABPFN_EMBEDDING_PATH:", TABPFN_EMBEDDING_PATH)
print("PROPOS_TABULAR_ADAPTER_PATH:", PROPOS_TABULAR_ADAPTER_PATH)
print("RUN_NAME:", RUN_NAME)
print("ARTIFACT_PATH:", ARTIFACT_PATH)
print("ASSIGNMENTS_PATH:", ASSIGNMENTS_PATH)
print("SUMMARY_PATH:", SUMMARY_PATH)

SCRIPT_DIR: /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model
TABPFN_EMBEDDING_PATH: /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/tabpfn_experiment_outputs/tabpfn_embeddings.parquet
PROPOS_TABULAR_ADAPTER_PATH: /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/ProPos-master/tabular_adapter.py
RUN_NAME: propos_tabpfn_20260609_142730
ARTIFACT_PATH: /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730.pt
ASSIGNMENTS_PATH: /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_assignments.parquet
SUMMARY_PATH: /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_summary.json


In [12]:
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)

# Test rapide
x_test = torch.randn(1000, 1000, device=DEVICE)
y_test = x_test @ x_test
print("Test tensor device:", y_test.device)

PyTorch: 2.10.0
MPS available: True
Using device: mps
Test tensor device: mps:0


In [13]:
if not TABPFN_EMBEDDING_PATH.exists():
    raise FileNotFoundError(
        f"Fichier TabPFN introuvable: {TABPFN_EMBEDDING_PATH}\n"
        "Tu dois d'abord générer tabpfn_embeddings.parquet."
    )

tabpfn_df = pd.read_parquet(TABPFN_EMBEDDING_PATH)

if "point_id" not in tabpfn_df.columns:
    raise RuntimeError("tabpfn_embeddings.parquet doit contenir une colonne point_id.")

tabpfn_df = tabpfn_df.copy()
tabpfn_df["point_id"] = pd.to_numeric(tabpfn_df["point_id"], errors="coerce")
tabpfn_df = tabpfn_df.dropna(subset=["point_id"]).copy()
tabpfn_df["point_id"] = tabpfn_df["point_id"].astype(np.int64)

tabpfn_df = tabpfn_df.drop_duplicates(subset=["point_id"], keep="first").reset_index(drop=True)

feature_cols = [col for col in tabpfn_df.columns if col != "point_id"]

if not feature_cols:
    raise RuntimeError("Aucune colonne embedding trouvée dans tabpfn_embeddings.parquet.")

for col in feature_cols:
    tabpfn_df[col] = pd.to_numeric(tabpfn_df[col], errors="coerce")

tabpfn_df[feature_cols] = (
    tabpfn_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

point_ids = tabpfn_df["point_id"].astype(np.int64).to_numpy()
x_tabpfn = tabpfn_df[feature_cols].to_numpy(dtype=np.float32)

print("TabPFN dataframe shape:", tabpfn_df.shape)
print("x_tabpfn shape:", x_tabpfn.shape)
print("Nombre de points:", len(point_ids))
print("Nombre de features TabPFN:", len(feature_cols))

tabpfn_df.head()

TabPFN dataframe shape: (5000, 513)
x_tabpfn shape: (5000, 512)
Nombre de points: 5000
Nombre de features TabPFN: 512


,point_id,tabpfn_z1,tabpfn_z2,tabpfn_z3,tabpfn_z4,tabpfn_z5,tabpfn_z6,tabpfn_z7,tabpfn_z8,tabpfn_z9,...,tabpfn_z503,tabpfn_z504,tabpfn_z505,tabpfn_z506,tabpfn_z507,tabpfn_z508,tabpfn_z509,tabpfn_z510,tabpfn_z511,tabpfn_z512
0,12666,0.231032,1.021180,1.152513,-0.922160,-0.279245,-0.326792,0.242194,1.508783,0.188240,...,-0.198951,-1.946219,-0.009572,1.312700,1.036481,0.223939,-0.428896,4.185240,-0.807237,-0.861083
1,12669,0.270116,0.574492,0.598429,-0.808564,-0.093174,-0.103679,0.243920,0.852235,0.148329,...,0.216722,-1.523250,0.105394,0.695460,1.057464,-0.452490,-0.393341,4.060300,-0.274252,-1.086566
2,12712,0.099727,-0.464563,-0.937437,-0.754460,0.451320,-0.252105,-0.432489,1.396306,0.130580,...,0.837341,0.246712,0.249083,-1.043237,-0.034535,0.115867,-0.231784,2.645921,1.110468,-0.593451
3,12727,0.336226,0.152188,0.202003,-0.396452,-0.798015,-0.454743,-0.491628,1.737358,0.151735,...,-0.210636,-0.699728,-0.212025,1.810661,-0.157290,0.142998,-0.200676,1.705691,-0.542633,-0.813913
4,12728,0.500314,-0.060534,-0.030531,-0.571617,-0.820358,-0.599913,-0.340717,1.520996,0.301821,...,-0.381308,-0.942314,-0.285185,1.910986,0.151870,0.232253,-0.325019,2.004017,-0.806723,-1.029181


In [8]:
pca_path = PROJECT_ROOT / "artifacts/experiments/tabpfn" / "pca_used_sample.parquet"
saint_path = PROJECT_ROOT / "artifacts/experiments/tabpfn" / "saint_used_sample.parquet"

pca_df = pd.read_parquet(pca_path)
saint_df = pd.read_parquet(saint_path)

ids_pca = set(pca_df["point_id"].astype(int))
ids_saint = set(saint_df["point_id"].astype(int))
ids_tabpfn = set(tabpfn_df["point_id"].astype(int))

print("PCA shape:", pca_df.shape)
print("SAINT shape:", saint_df.shape)
print("TabPFN shape:", tabpfn_df.shape)

print("Même PCA / SAINT:", ids_pca == ids_saint)
print("Même PCA / TabPFN:", ids_pca == ids_tabpfn)
print("Même SAINT / TabPFN:", ids_saint == ids_tabpfn)

print("Nombre de points communs:", len(ids_pca & ids_saint & ids_tabpfn))

PCA shape: (5000, 9)
SAINT shape: (5000, 9)
TabPFN shape: (5000, 513)
Même PCA / SAINT: True
Même PCA / TabPFN: True
Même SAINT / TabPFN: True
Nombre de points communs: 5000


In [14]:

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_tabpfn).astype(np.float32)

print("x_scaled shape:", x_scaled.shape)
print("mean:", float(np.mean(x_scaled)))
print("std:", float(np.std(x_scaled)))

x_scaled shape: (5000, 512)
mean: 1.3627111661307367e-09
std: 1.0


In [15]:
def load_propos_adapter():
    if not PROPOS_TABULAR_ADAPTER_PATH.exists():
        raise FileNotFoundError(
            f"Adaptateur ProPos introuvable: {PROPOS_TABULAR_ADAPTER_PATH}"
        )

    master_dir = str(PROPOS_MASTER_DIR)

    if master_dir not in sys.path:
        sys.path.insert(0, master_dir)

    spec = importlib.util.spec_from_file_location(
        "propos_master_tabular_adapter",
        PROPOS_TABULAR_ADAPTER_PATH,
    )

    if spec is None or spec.loader is None:
        raise RuntimeError("Impossible de charger ProPos-master/tabular_adapter.py")

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    if not hasattr(module, "train_tabular_propos"):
        raise RuntimeError("tabular_adapter.py ne contient pas train_tabular_propos.")

    return module


adapter = load_propos_adapter()

print("ProPos adapter chargé correctement.")
print("Fonction disponible:", hasattr(adapter, "train_tabular_propos"))

ProPos adapter chargé correctement.
Fonction disponible: True


In [16]:
def train_propos_tabpfn(
    x: np.ndarray,
    k: int,
    epochs: int,
    device: str,
    seed: int,
) -> dict:
    fn = adapter.train_tabular_propos

    attempts = [
        lambda current_device: fn(
            x,
            num_clusters=int(k),
            epochs=int(epochs),
            learning_rate=float(LEARNING_RATE),
            weight_decay=float(WEIGHT_DECAY),
            device=current_device,
            seed=seed,
        ),
        lambda current_device: fn(
            x,
            num_clusters=int(k),
            epochs=int(epochs),
            device=current_device,
            seed=seed,
        ),
        lambda current_device: fn(
            x,
            k=int(k),
            epochs=int(epochs),
            device=current_device,
            seed=seed,
        ),
    ]

    devices_to_try = [device]

    # Sécurité : si MPS échoue, on réessaie sur CPU.
    if device != "cpu":
        devices_to_try.append("cpu")

    last_error = None

    for current_device in devices_to_try:
        for attempt in attempts:
            try:
                print(f"Essai ProPos avec device={current_device}")
                result = attempt(current_device)

                if not isinstance(result, dict):
                    raise RuntimeError("train_tabular_propos n'a pas retourné un dictionnaire.")

                return result

            except TypeError as exc:
                last_error = exc
                continue

            except Exception as exc:
                last_error = exc
                print(f"Échec avec device={current_device}: {exc}")
                break

    raise RuntimeError(f"Impossible d'appeler train_tabular_propos: {last_error}")


start = time.perf_counter()

result = train_propos_tabpfn(
    x=x_scaled,
    k=K,
    epochs=EPOCHS,
    device=DEVICE,
    seed=RANDOM_STATE,
)

duration_s = time.perf_counter() - start

if "labels" not in result:
    raise RuntimeError("Résultat ProPos invalide: clé labels absente.")

labels = np.asarray(result["labels"], dtype=int)

print("Entraînement terminé.")
print("Durée secondes:", round(duration_s, 2))
print("Labels shape:", labels.shape)
print("Clusters:", sorted(set(labels.tolist())))
print("Distribution:", dict(zip(*np.unique(labels, return_counts=True))))

Essai ProPos avec device=mps
Entraînement terminé.
Durée secondes: 22.6
Labels shape: (5000,)
Clusters: [0, 1, 2, 3, 4]
Distribution: {np.int64(0): np.int64(723), np.int64(1): np.int64(2119), np.int64(2): np.int64(720), np.int64(3): np.int64(643), np.int64(4): np.int64(795)}


In [17]:
def compute_valid_metrics(x: np.ndarray, labels: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=int)

    noise_mask = labels == -1
    valid_mask = ~noise_mask

    valid_x = x[valid_mask]
    valid_labels = labels[valid_mask]

    unique_clusters = sorted({int(v) for v in valid_labels.tolist()})
    n_clusters = len(unique_clusters)
    noise_points = int(noise_mask.sum())

    metrics = {
        "n_points": int(len(labels)),
        "n_features": int(x.shape[1]),
        "n_clusters": int(n_clusters),
        "noise_points": int(noise_points),
        "noise_ratio": float(noise_points / len(labels)) if len(labels) else 0.0,
        "silhouette": None,
        "davies_bouldin": None,
        "calinski_harabasz": None,
    }

    if n_clusters < 2 or len(valid_x) <= n_clusters:
        return metrics

    metrics["silhouette"] = float(
        silhouette_score(
            valid_x,
            valid_labels,
            sample_size=min(5000, len(valid_x)),
            random_state=RANDOM_STATE,
        )
    )

    metrics["davies_bouldin"] = float(
        davies_bouldin_score(valid_x, valid_labels)
    )

    metrics["calinski_harabasz"] = float(
        calinski_harabasz_score(valid_x, valid_labels)
    )

    return metrics


metrics = compute_valid_metrics(x_tabpfn, labels)

metrics

{'n_points': 5000,
 'n_features': 512,
 'n_clusters': 5,
 'noise_points': 0,
 'noise_ratio': 0.0,
 'silhouette': 0.4706399440765381,
 'davies_bouldin': 1.3714880956762734,
 'calinski_harabasz': 3598.123332509475}

In [18]:
assignments_df = pd.DataFrame(
    {
        "point_id": point_ids.astype(np.int64),
        "cluster_id": labels.astype(int),
    }
)

if "pointConfidence" in result:
    try:
        assignments_df["assignment_confidence"] = np.asarray(
            result["pointConfidence"],
            dtype=float,
        )
    except Exception:
        pass

assignments_df.to_parquet(ASSIGNMENTS_PATH, index=False)

print("Assignments sauvegardés:")
print(ASSIGNMENTS_PATH)

assignments_df.head()

Assignments sauvegardés:
/Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_assignments.parquet


,point_id,cluster_id,assignment_confidence
0,12666,3,0.893122
1,12669,3,0.825009
2,12712,0,0.917944
3,12727,2,0.741901
4,12728,2,0.769076


In [19]:
artifact = {
    "modelState": result.get("modelState"),
    "modelConfig": result.get("modelConfig"),
    "centroids": result.get("centroids"),
    "trainingLoss": result.get("trainingLoss"),
    "epochs": result.get("epochs"),
    "featureMetadata": {
        "representation": "tabpfn",
        "input_path": str(TABPFN_EMBEDDING_PATH),
        "feature_columns": feature_cols,
        "n_points": int(x_tabpfn.shape[0]),
        "n_features": int(x_tabpfn.shape[1]),
        "standardized": True,
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
    },
    "purpose": "tabpfn_propos_reference",
    "artifactPath": str(ARTIFACT_PATH.resolve()),
    "savedAt": datetime.now().isoformat(),
}

torch.save(artifact, ARTIFACT_PATH)

print("Artefact ProPos TabPFN sauvegardé:")
print(ARTIFACT_PATH)

Artefact ProPos TabPFN sauvegardé:
/Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730.pt


In [20]:
summary = {
    "run_name": RUN_NAME,
    "method": "TabPFN + ProPos",
    "representation": "tabpfn",
    "clustering": "propos",
    "input_embedding_path": str(TABPFN_EMBEDDING_PATH),
    "artifact_path": str(ARTIFACT_PATH),
    "assignments_path": str(ASSIGNMENTS_PATH),
    "summary_path": str(SUMMARY_PATH),
    "k": int(K),
    "epochs": int(EPOCHS),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "device_requested": DEVICE,
    "duration_seconds": float(duration_s),
    "metrics": metrics,
    "cluster_distribution": {
        str(int(label)): int(count)
        for label, count in zip(*np.unique(labels, return_counts=True))
    },
    "created_at": datetime.now().isoformat(),
}

SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Résumé sauvegardé:")
print(SUMMARY_PATH)

summary

Résumé sauvegardé:
/Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_summary.json


{'run_name': 'propos_tabpfn_20260609_142730',
 'method': 'TabPFN + ProPos',
 'representation': 'tabpfn',
 'clustering': 'propos',
 'input_embedding_path': '/Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/tabpfn_experiment_outputs/tabpfn_embeddings.parquet',
 'artifact_path': '/Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730.pt',
 'assignments_path': '/Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_assignments.parquet',
 'summary_path': '/Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_summary.json',
 'k': 5,
 'epochs': 100,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'device_requested': 'mps',
 'duration_seconds': 22.602196499996353,
 'metrics': {'n_points': 5000,
  'n_features': 512,
  'n_clusters': 5,
  'noise_points': 0,
  'noise_ratio': 0.0,
  'silhouette': 0.4706399440765381,
  'davies_bouldin': 1.3714880956762734,
  'calinski_har

In [21]:
print("Fichiers ProPos TabPFN générés:")

for path in sorted(PROPOS_CACHE_DIR.glob("propos_tabpfn_*")):
    print(path.name)

print("\nDerniers fichiers:")
print("Artifact:", ARTIFACT_PATH.exists(), ARTIFACT_PATH)
print("Assignments:", ASSIGNMENTS_PATH.exists(), ASSIGNMENTS_PATH)
print("Summary:", SUMMARY_PATH.exists(), SUMMARY_PATH)

Fichiers ProPos TabPFN générés:
propos_tabpfn_20260609_142730.pt
propos_tabpfn_20260609_142730_assignments.parquet
propos_tabpfn_20260609_142730_summary.json

Derniers fichiers:
Artifact: True /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730.pt
Assignments: True /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_assignments.parquet
Summary: True /Users/ayatbouhrir/Desktop/STAAGE/test/Projet/model/.propos_cache/propos_tabpfn_20260609_142730_summary.json
